# 05 - Lakeflow Jobs: Orquestación, Monitoreo y Linaje

## Crear Job: `BNCR Workshop Pipeline - <usuario>`

### Tareas
1. **Carga_Incremental** (Notebook: `99_incremental`)
2. **Pipeline_Medallion** (Pipeline LDP) → depends on 1
3. **Consulta_Gold** (SQL) → depends on 2


In [0]:
%run "./00 - Setup/00_variables"


In [0]:
sql_gold = f"""
SELECT fecha_transaccion, nombre_sucursal, provincia, total_transacciones, monto_total
FROM {catalog_name}.{schema_gold}.resumen_diario_sucursal
ORDER BY monto_total DESC LIMIT 100
"""
print(sql_gold)


## Triggers recomendados

| Opción | Consumo |
|--------|--------|
| Scheduled 30min | Alto continuo |
| Continuous | Máximo |
| File arrival | Por evento |

## Monitoreo
- Timeline view
- Camino crítico
- Linaje por ejecución


###Genie Code 

Prompt: 
Crea un job, usando este notebook como ejemplo 

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import Task, NotebookTask, PipelineTask, SqlTask, SqlTaskQuery, TaskDependency
from databricks.sdk.service.sql import CreateQueryRequestQuery

w = WorkspaceClient()

# ---- Configuración ----
job_name        = f"BNCR Workshop Pipeline - {user_suffix}"
notebook_path   = "/Workspace/Users/rico.martinez@databricks.com/Latam_resources_spanish/Data_Engineering/00 - Setup/99_incremental"
pipeline_id     = "147799ee-abfb-4f5f-ab5f-8af4fc3ad7fb"   # <-- ajuste si su pipeline es diferente
warehouse_id    = "18479908b6be4949"                        # Serverless Starter Warehouse

# ---- 1. Crear query SQL guardada para la tarea Gold ----
gold_query = w.queries.create(query=CreateQueryRequestQuery(
    display_name=f"BNCR Gold Resumen - {user_suffix}",
    warehouse_id=warehouse_id,
    query_text=sql_gold,
))
print(f"✅ Query creada: {gold_query.id}")

# ---- 2. Crear Job con 3 tareas + dependencias ----
job = w.jobs.create(
    name=job_name,
    tasks=[
        Task(
            task_key="Carga_Incremental",
            description="Carga archivos incrementales al volumen",
            notebook_task=NotebookTask(notebook_path=notebook_path),
        ),
        Task(
            task_key="Pipeline_Medallion",
            description="Ejecuta el pipeline LDP (bronze → silver → gold)",
            pipeline_task=PipelineTask(pipeline_id=pipeline_id),
            depends_on=[TaskDependency(task_key="Carga_Incremental")],
        ),
        Task(
            task_key="Consulta_Gold",
            description="Consulta resumen diario de la capa gold",
            sql_task=SqlTask(
                query=SqlTaskQuery(query_id=gold_query.id),
                warehouse_id=warehouse_id,
            ),
            depends_on=[TaskDependency(task_key="Pipeline_Medallion")],
        ),
    ],
)

print(f"✅ Job creado: {job.job_id}")
print(f"   Nombre : {job_name}")
print(f"   Tareas : Carga_Incremental → Pipeline_Medallion → Consulta_Gold")
print(f"   URL    : /jobs/{job.job_id}")